# Production-Style Demo: IMDb Review → Prediction (Kafka → Bronze/Silver/Gold → UI)

This notebook documents a production-style demo for `text-ml-platform`.

**Why this matters:** Modern ML systems need reproducible pipelines, scalable ingestion (Kafka), ACID storage (Iceberg), and flexible inference (sync + async). This demo shows how to combine them for sentiment classification—a pattern that extends to other text tasks (NER, summarization, custom classifiers) and production MLOps.

**Potential usage:** Sentiment/classification demos, MLOps learning, embedding+classifier stacks, async inference at scale, data engineering patterns (medallion, Kafka, Iceberg).

**Steps:**

1. Stream data (train/test) into Kafka—from the IMDb dataset or from **local Ollama**–generated reviews.
2. Land into bronze, clean into silver.
3. Generate BERT embeddings into Iceberg gold tables.
4. Train a simple classifier on gold embeddings.
5. Run a prediction service + async worker.
6. Use the Streamlit UI to enter a review and get `positive/negative`.

You will run long-lived services (`bronze_consumer`, `predict_service`, `inference_worker`, `streamlit`) in separate terminals. For a fully automated run, use Docker and the `prepopulate_imdb` service (see README).

## Architecture: How It All Works

The platform follows a **medallion architecture** (bronze → silver → gold) with Kafka for ingestion and Iceberg for the gold layer. This diagram summarizes the data flow:

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│  SOURCE                    │  INGESTION        │  STORAGE (MinIO)  │  GOLD (Iceberg) │
├────────────────────────────┼───────────────────┼───────────────────┼─────────────────┤
│  • IMDb (Hugging Face)     │  Producer ──────► │  Kafka            │                 │
│  • Ollama (LLM-generated)  │                   │      │            │                 │
└────────────────────────────┴───────────────────┴───────┼──────────┴─────────────────┘
                                                         │
                                                         ▼
┌─────────────────────────────────────────────────────────────────────────────────┐
│  Bronze Consumer    ──────►  bronze/imdb/<split>/*.jsonl  (raw JSONL)            │
│  Silver Job         ──────►  silver/imdb/<split>/*.jsonl  (cleaned, deduplicated)│
│  Embedding Job      ──────►  imdb.gold_train, imdb.gold_test (BERT embeddings)   │
│  Train Classifier   ◄──────  gold_train → models/sentiment_logreg.joblib         │
└─────────────────────────────────────────────────────────────────────────────────┘
                                                         │
┌─────────────────────────────────────────────────────────────────────────────────┐
│  INFERENCE                                                                       │
│  • Sync:  UI → Predict API (HTTP) → BERT + classifier → response                 │
│  • Async: UI → Kafka → Inference Worker → bronze/silver/gold_inference           │
│           → imdb.predictions → UI polls Iceberg for result                       │
└─────────────────────────────────────────────────────────────────────────────────┘
```

**Why this design?**

| Component | Purpose |
|-----------|---------|
| **Kafka** | Decouples producers and consumers; enables batch and real-time streaming. |
| **Bronze** | Raw, immutable landing zone. |
| **Silver** | Cleaned, deduplicated data ready for feature engineering. |
| **Gold (Iceberg)** | Model-ready embeddings with ACID, time travel, schema evolution. |
| **Sync vs async** | Sync for low-latency UI; async for high-throughput, fire-and-forget, and full lineage (bronze → silver → gold → predictions). |

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

# Ensure local imports work in notebooks.
os.environ.setdefault("PYTHONPATH", str(PROJECT_ROOT))
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

## Prerequisites

- Docker Compose running Kafka + MinIO:
  ```bash
  docker compose -f docker/docker-compose.yml up -d
  ```
- Python deps installed (from `requirements.txt`). For Iceberg gold tables and the prediction service you need `pyiceberg` (e.g. `pip install pyiceberg`).
- Run all terminal commands from the **project root** so default paths (e.g. `models/sentiment_logreg.joblib`) resolve correctly.

**Alternative (all-in-one Docker):** You can skip the manual steps and run everything in Docker:

```bash
# Start infra + demo services
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml up -d --build

# Prepopulate IMDb data and train (one-shot: producer → bronze → silver → embedding → train)
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml run --rm prepopulate_imdb

# Verify Iceberg tables exist
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml run --rm gold_iceberg_test
```

Then open the Streamlit UI at http://localhost:8501 and the Predict API at http://localhost:8002. Use `docker-compose.demo.yml` instead of `docker-compose.demo.gpu.yml` for CPU-only.

## Step 1: Start consumers/services in separate terminals

### Terminal A: Kafka → Bronze consumer (writes `bronze/imdb/<split>/`)

**Batch mode (consumer exits after ingesting):** Use `--limit` so the consumer stops once it has read the produced messages:
```bash
# Run producer first (Step 2), then run consumer with limit = total produced (e.g. 200 train + 200 test = 400)
python -m src.ingestion.bronze_consumer --batch-size 50 --limit 400 --split train
```

**Continuous mode (leave running):** Omit `--limit`; the consumer runs until interrupted.
```bash
python -m src.ingestion.bronze_consumer --batch-size 50 --split train
```

### Terminal B: (later) Prediction service (FastAPI)
```bash
python -m src.inference.predict_service --model-path models/sentiment_logreg.joblib --port 8000 --iceberg-write
```

This powers the synchronous UI mode. Train the classifier first (Step 5) so `models/sentiment_logreg.joblib` exists; the model path is resolved from the project root so you can run from any directory.

### Terminal C: (later) Async inference worker (Kafka → bronze/silver/gold + predictions)
```bash
python -m src.inference.inference_worker --classifier-model models/sentiment_logreg.joblib --iceberg --iceberg-namespace imdb --iceberg-table gold_inference
```

This powers the asynchronous UI mode. The worker consumes topic **`imdb-inference`** (`KAFKA_INFERENCE_TOPIC`), not `imdb-reviews`, so bulk prepopulate traffic does not delay UI requests. Async mode is slower than sync by design (full medallion + Iceberg per request).

## Step 2: Stream train/test into Kafka

Run these commands from the project root (any terminal). Use a small `--limit` for a quick demo. **Do not use `--no-shuffle`**: shuffling ensures both positive and negative reviews are streamed, so the classifier can be trained on both classes.

In [ ]:
import subprocess

def run(cmd):
    print("\nRunning:", " ".join(map(str, cmd)))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print("STDERR:\n", res.stderr)
    return res.returncode

# Stream train (shuffle so both positive/negative labels appear; required for training)
run(["python","-m","src.ingestion.producer","--mode","batch","--split","train","--limit","200"])

# Stream test
run(["python","-m","src.ingestion.producer","--mode","batch","--split","test","--limit","200"])

Stop Terminal A only when bronze has received enough data for the demo.

### Option: Generate reviews with local Ollama

Instead of (or in addition to) streaming from the IMDb dataset, you can generate synthetic movie reviews with **local Ollama**. The generator uses the same OpenAI-compatible `/v1/chat/completions` API and publishes to the same Kafka topic, so the bronze consumer and downstream pipeline are unchanged.

1. Install and start [Ollama](https://ollama.com), then pull a model: `ollama pull llama3.2`
2. With Terminal A (bronze consumer) running, run from project root:
   ```bash
   python -m src.llm.generate_reviews_client --api-base http://localhost:11434/v1 --model llama3.2 --sentiment positive --count 25 --split train
   python -m src.llm.generate_reviews_client --api-base http://localhost:11434/v1 --model llama3.2 --sentiment negative --count 25 --split train
   ```
   Generate both sentiments so the training set has both classes.

In [ ]:
# Optional: generate synthetic reviews with local Ollama (skip if you used Step 2 producer above)
OLLAMA_BASE = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2"  # or mistral, llama2, etc.

run(["python", "-m", "src.llm.generate_reviews_client",
     "--api-base", OLLAMA_BASE, "--model", OLLAMA_MODEL,
     "--sentiment", "positive", "--count", "25", "--split", "train"])
run(["python", "-m", "src.llm.generate_reviews_client",
     "--api-base", OLLAMA_BASE, "--model", OLLAMA_MODEL,
     "--sentiment", "negative", "--count", "25", "--split", "train"])

## Step 3: Bronze → Silver (run separately for train/test)

```bash
python -m src.transformation.silver_job --bronze-prefix bronze/imdb/train/ --silver-prefix silver/imdb/train/
python -m src.transformation.silver_job --bronze-prefix bronze/imdb/test/  --silver-prefix silver/imdb/test/
```

In [ ]:
run(["python","-m","src.transformation.silver_job","--bronze-prefix","bronze/imdb/train/","--silver-prefix","silver/imdb/train/"])
run(["python","-m","src.transformation.silver_job","--bronze-prefix","bronze/imdb/test/","--silver-prefix","silver/imdb/test/"])


## Step 4: (Optional) Fine-tune BERT on silver data

For better sentiment performance, fine-tune BERT on IMDb before embedding. Skip this for a quicker demo; the embedding job will use the default pre-trained model.

```bash
python -m src.training.finetune_bert --silver-train-prefix silver/imdb/train/ --silver-test-prefix silver/imdb/test/ --output-dir models/bert_sentiment_imdb
```

## Step 5: Silver → Gold embeddings (Iceberg tables per split)

Requires `pyiceberg` installed. Run **both** commands so you have `imdb.gold_train` and `imdb.gold_test` for training and evaluation.

If you ran Step 4 (finetune), add `--bert-path models/bert_sentiment_imdb` to use the fine-tuned model.

```bash
# Train embeddings -> imdb.gold_train
python -m src.features.embedding_job --silver-prefix silver/imdb/train/ --iceberg --iceberg-namespace imdb --iceberg-table gold_train --bert-path models/bert_sentiment_imdb

# Test embeddings -> imdb.gold_test
python -m src.features.embedding_job --silver-prefix silver/imdb/test/  --iceberg --iceberg-namespace imdb --iceberg-table gold_test  --bert-path models/bert_sentiment_imdb
```

(Omit `--bert-path` if you skipped Step 4.)

In [ ]:
# Optional: fine-tune BERT (run before embedding for better accuracy)
run(["python","-m","src.training.finetune_bert",
     "--silver-train-prefix","silver/imdb/train/",
     "--silver-test-prefix","silver/imdb/test/",
     "--output-dir","models/bert_sentiment_imdb"])

In [ ]:
# Add --bert-path models/bert_sentiment_imdb if you ran finetune_bert (Step 4)
run(["python","-m","src.features.embedding_job",
     "--silver-prefix","silver/imdb/train/",
     "--iceberg","--iceberg-namespace","imdb","--iceberg-table","gold_train",
     "--bert-path","models/bert_sentiment_imdb"])

run(["python","-m","src.features.embedding_job",
     "--silver-prefix","silver/imdb/test/",
     "--iceberg","--iceberg-namespace","imdb","--iceberg-table","gold_test",
     "--bert-path","models/bert_sentiment_imdb"])

## Step 5: Train classifier (logistic regression on embeddings)

Uses `imdb.gold_train` for training and `imdb.gold_test` for evaluation (both must exist from Step 4). Pass `--test-iceberg-identifier imdb.gold_test` so the trainer evaluates on the separate test table and reports test accuracy. The trainer needs at least two label classes in the training data; if you see "only one class" then stream more data or check silver/gold have both positive and negative reviews.

```bash
python -m src.training.train_classifier \
  --iceberg-identifier imdb.gold_train \
  --test-iceberg-identifier imdb.gold_test \
  --train-split train \
  --test-split test \
  --model-out models/sentiment_logreg.joblib
```

## Docker: Restart, Prepopulate, Monitoring

**Restart everything:**
```bash
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml down
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml up -d --build
# Re-prepopulate if you cleared data
docker compose -f docker/docker-compose.yml -f docker/docker-compose.demo.gpu.yml run --rm prepopulate_imdb
```

**Prometheus metrics:** With `prometheus_client` installed, the Predict API exposes metrics at http://localhost:8002/metrics (e.g. `predict_requests_total`, `predict_latency_seconds`, `inference_consumed_total`). Scrape with Prometheus or `curl http://localhost:8002/metrics`.

**Configuration:** All runtime config lives in `src/config.py` and is overridden by env vars (`KAFKA_BOOTSTRAP_SERVERS`, `S3_ENDPOINT_URL`, `ICEBERG_CATALOG_DB`, `PREDICT_API_URL`, etc.). Docker Compose sets these for containers.

**Tests:** Run `pytest tests/ -v` from the project root. Use `--ignore=tests/test_gold_iceberg_table.py` to skip the Iceberg integration test when MinIO isn't running.

In [ ]:
run(["python","-m","src.training.train_classifier",
     "--iceberg-identifier","imdb.gold_train",
     "--test-iceberg-identifier","imdb.gold_test",
     "--train-split","train",
     "--test-split","test",
     "--model-out","models/sentiment_logreg.joblib",
     "--max-records","0"])

## Step 7: Verify Iceberg tables + then start the UI

Run this verification to ensure `imdb.gold_train` and `imdb.gold_test` exist and have rows. If either table is missing, re-run the corresponding embedding job in Step 4.

In [ ]:
from src.utils.iceberg_catalog import get_iceberg_catalog

catalog = get_iceberg_catalog()
tables = catalog.list_tables("imdb")
print("Tables in imdb namespace:", tables)

for name in ("gold_train", "gold_test"):
    try:
        tbl = catalog.load_table(f"imdb.{name}")
        print(f"{name} rows:", tbl.scan().count())
    except Exception as e:
        print(f"{name}: not found ({e})")

## Step 8: Run Streamlit UI

```bash
streamlit run src/ui/streamlit_app.py
```

The UI supports:
- `sync`: call `predict_service` directly
- `async`: send to Kafka and poll `imdb.predictions` via the inference worker

It also visualizes the entered review in the PCA projection of gold embeddings.